# HC-SMoE 1-Click: Qwen ZipIt Merge

Run **Qwen1.5-MoE-A2.7B** ZipIt merge in a notebook (single GPU). No bash/accelerate needed.

**After merge**, load in your benchmark with:
```python
AutoModelForCausalLM.from_pretrained(output_path, trust_remote_code=True)
```

## 1. Setup (run once)

Set `REPO_DIR` to the path of this HC-SMoE repo (e.g. Colab: `/content/drive/MyDrive/.../HC-SMoE`).

In [ ]:
import os
import sys

# HC-SMoE repo root. If you opened this notebook from HC-SMoE/experiment/, use "..".
# Colab: set to your path, e.g. "/content/drive/MyDrive/.../HC-SMoE"
REPO_DIR = os.path.abspath("..")
if not os.path.isdir(os.path.join(REPO_DIR, "hcsmoe")):
    REPO_DIR = os.path.abspath(".")
OUTPUT_DIR = os.path.join(REPO_DIR, "saved_models", "qwen", "zipit", "groups_45_ing_act")

os.makedirs(OUTPUT_DIR, exist_ok=True)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)
print("REPO_DIR:", REPO_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)

In [ ]:
# Optional: install deps (uncomment in Colab if needed)
# !pip install -r requirements.txt
# !pip install lm-eval

## 2. Run merge (1-click)

Same config as `experiment/qwen/run_zipit.sh` with `NUM_GROUPS=45`, `INGREDIENT=act`. No evaluation (`task="no"`).

In [ ]:
import importlib.util
spec = importlib.util.spec_from_file_location(
    "merging_qwen",
    os.path.join(REPO_DIR, "hcsmoe", "merging-qwen.py"),
)
mod = importlib.util.module_from_spec(spec)
spec.loader.exec_module(mod)

mod.run_hcsmoe(
    task="no",
    model_name="Qwen/Qwen1.5-MoE-A2.7B-Chat",
    dominant="no",
    similarity_base="expert-output",
    cluster="hierarchical",
    linkage="average",
    merge="zipit",
    mode="normal",
    ingredient="act",
    num_average_groups=45,
    n_sentences=32,
    train_batch_size=2,
    start_layer=0,
    partition=1,
    data_limit=1_000_000,
    output_path=OUTPUT_DIR,
    result_path=os.path.join(OUTPUT_DIR, "eval.txt"),
)